# AEZ-26 Phase 2E — Native-Time Response Alignment

This notebook aligns the **39 locked Phase 2C v3 flash-drought events** with:

- MOD16A2GF ET/PET
- MOD13Q1 NDVI/EVI
- GRACE/GRACE-FO TWSA
- JRC Monthly Surface Water

## Non-negotiable rules

- no daily interpolation of response products;
- no modification of flash-drought labels;
- MOD13 remains anchored to its native composite start date;
- GRACE remains TWSA, not direct groundwater;
- JRC surface-water area remains area, not storage volume;
- all QC/coverage fields are retained.

All Phase 2E tables, figures, and one ZIP audit package are saved automatically to Google Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil
from pandas.tseries.offsets import MonthEnd

BASE_DIR = Path('/content/drive/MyDrive/AEZ26_Final_Research_Data')
PHASE2_DIR = BASE_DIR / 'Phase2'
PHASE2C_DIR = PHASE2_DIR / 'Phase2C_v3'
PHASE2D_DIR = PHASE2_DIR / 'Phase2D'
PHASE2E_DIR = PHASE2_DIR / 'Phase2E'
FIG_DIR = PHASE2E_DIR / 'Figures'

PHASE2E_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Authoritative event inputs
EVENT_FILE = PHASE2C_DIR / 'AEZ26_Flash_Drought_Event_Catalogue_Primary_2000_2025.csv'
PENTAD_FILE = PHASE2C_DIR / 'AEZ26_Pentad_Hydroclimate_Indicators_2000_2025.csv'
DRIVER_FILE = PHASE2D_DIR / 'AEZ26_Phase2D_Event_Driver_Characterization.csv'

# Authoritative Phase-1 response datasets
MOD16_FILE = BASE_DIR / 'AEZ26_MODIS_ET_PET_QC_2000_2025.csv'
MOD13_FILE = BASE_DIR / 'AEZ26_MODIS_NDVI_EVI_QC_2000_2025.csv'
GRACE_FILE = BASE_DIR / 'AEZ26_GRACE_Mascon_TWSA_Uncertainty.csv'
JRC_FILE = BASE_DIR / 'AEZ26_JRC_SurfaceWater_Monthly_QC_2000_2021.csv'

required = [
    EVENT_FILE,
    PENTAD_FILE,
    DRIVER_FILE,
    MOD16_FILE,
    MOD13_FILE,
    GRACE_FILE,
    JRC_FILE
]

for p in required:
    print(p.name, '->', p.exists())
    if not p.exists():
        raise FileNotFoundError(f'Missing required input: {p}')

print()
print('Phase 2E output folder:', PHASE2E_DIR)


In [ ]:
events = pd.read_csv(
    EVENT_FILE,
    parse_dates=[
        'anchor_mid_date',
        'onset_start_date',
        'onset_mid_date',
        'threshold_crossing_mid_date',
        'event_end_date',
        'termination_mid_date'
    ]
).sort_values('onset_start_date').reset_index(drop=True)

pentad = pd.read_csv(
    PENTAD_FILE,
    parse_dates=[
        'pentad_start',
        'pentad_end',
        'pentad_mid_date'
    ]
).sort_values('pentad_start').reset_index(drop=True)

drivers = pd.read_csv(
    DRIVER_FILE,
    parse_dates=['onset_mid_date']
)

mod16 = pd.read_csv(
    MOD16_FILE,
    parse_dates=[
        'composite_start_date',
        'composite_end_date'
    ]
)

mod13 = pd.read_csv(
    MOD13_FILE,
    parse_dates=['composite_start_date']
)

grace = pd.read_csv(
    GRACE_FILE,
    parse_dates=[
        'native_date',
        'native_end_date_if_available'
    ]
)

jrc = pd.read_csv(
    JRC_FILE,
    parse_dates=['date']
)

# Strict input gates
assert len(events) == 39
assert events['event_id'].is_unique
assert len(pentad) == 1898
assert pentad['pentad_label'].is_unique

assert len(drivers) == 39
assert set(drivers['event_id']) == set(events['event_id'])

assert len(mod16) == 1196
assert len(mod13) == 595
assert len(grace) == 238
assert len(jrc) == 264

assert mod16['composite_start_date'].is_unique
assert mod13['composite_start_date'].is_unique
assert grace['native_system_index'].is_unique
assert jrc['native_system_index'].is_unique

print('Input row-count / uniqueness gate: PASS')


## 1. Event chronology and helper functions


In [ ]:
pentad_lookup = pentad.set_index('pentad_label')
driver_lookup = drivers.set_index('event_id')

events['threshold_crossing_start_date'] = (
    events['threshold_crossing_pentad']
    .map(pentad_lookup['pentad_start'])
)

events['threshold_crossing_end_date'] = (
    events['threshold_crossing_pentad']
    .map(pentad_lookup['pentad_end'])
)

events['primary_driver_class'] = (
    events['event_id']
    .map(driver_lookup['primary_driver_class'])
)

events['onset_season'] = (
    events['event_id']
    .map(driver_lookup['onset_season'])
)

events['P_standardized_anomaly_z'] = (
    events['event_id']
    .map(driver_lookup['P_standardized_anomaly_z'])
)

events['PE_standardized_anomaly_z'] = (
    events['event_id']
    .map(driver_lookup['PE_standardized_anomaly_z'])
)

assert events['threshold_crossing_end_date'].notna().all()
assert events['primary_driver_class'].notna().all()


def overlap_days(a_start, a_end, b_start, b_end):
    a_start = pd.Timestamp(a_start)
    a_end = pd.Timestamp(a_end)
    b_start = pd.Timestamp(b_start)
    b_end = pd.Timestamp(b_end)

    start = max(a_start, b_start)
    end = min(a_end, b_end)

    if end < start:
        return 0

    return int((end - start).days + 1)


event_intervals = [
    (
        row.event_id,
        pd.Timestamp(row.onset_start_date),
        pd.Timestamp(row.event_end_date)
    )
    for row in events.itertuples(index=False)
]


def overlaps_other_event(event_id, interval_start, interval_end):
    for other_id, other_start, other_end in event_intervals:

        if other_id == event_id:
            continue

        if overlap_days(
            interval_start,
            interval_end,
            other_start,
            other_end
        ) > 0:
            return 1

    return 0


def point_inside_other_event(event_id, date):
    date = pd.Timestamp(date)

    for other_id, other_start, other_end in event_intervals:

        if other_id == event_id:
            continue

        if other_start <= date <= other_end:
            return 1

    return 0


print('Event chronology helper gate: PASS')


## 2. MOD16A2GF native-interval linkage

Operational context window:
- 32 days before onset
- 64 days after event end

This window is for packaging context only and is not an inferential lag threshold.


In [ ]:
# Verify actual composite duration against exported composite_days.
mod16_duration_check = (
    (mod16['composite_end_date'] - mod16['composite_start_date'])
    .dt.days
    + 1
)

assert np.allclose(
    mod16_duration_check,
    mod16['composite_days'],
    equal_nan=False
)

mod16_rows = []

for ev in events.itertuples(index=False):

    context_start = (
        ev.onset_start_date
        - pd.Timedelta(days=32)
    )

    context_end = (
        ev.event_end_date
        + pd.Timedelta(days=64)
    )

    subset = mod16.loc[
        (mod16['composite_end_date'] >= context_start)
        &
        (mod16['composite_start_date'] <= context_end)
    ].copy()

    for obs in subset.itertuples(index=False):

        start = obs.composite_start_date
        end = obs.composite_end_date
        midpoint = start + (end - start) / 2
        native_days = int((end - start).days + 1)

        intensification_overlap = overlap_days(
            start,
            end,
            ev.onset_start_date,
            ev.threshold_crossing_end_date
        )

        persistence_start = (
            ev.threshold_crossing_end_date
            + pd.Timedelta(days=1)
        )

        persistence_overlap = 0

        if persistence_start <= ev.event_end_date:
            persistence_overlap = overlap_days(
                start,
                end,
                persistence_start,
                ev.event_end_date
            )

        event_overlap = (
            intensification_overlap
            + persistence_overlap
        )

        if event_overlap > 0:

            if (
                intensification_overlap > 0
                and persistence_overlap > 0
            ):
                phase = 'mixed_intensification_persistence'

            elif intensification_overlap > 0:
                phase = 'intensification_overlap'

            else:
                phase = 'persistence_overlap'

        elif end < ev.onset_start_date:
            phase = 'pre_event_context'

        else:
            phase = 'post_event_context'

        mod16_rows.append({
            'event_id': ev.event_id,
            'primary_driver_class':
                ev.primary_driver_class,
            'onset_season':
                ev.onset_season,

            'onset_start_date':
                ev.onset_start_date,
            'threshold_crossing_end_date':
                ev.threshold_crossing_end_date,
            'event_end_date':
                ev.event_end_date,

            'native_start_date':
                start,
            'native_end_date':
                end,
            'native_mid_date':
                midpoint,
            'native_days':
                native_days,

            'alignment_phase':
                phase,

            'overlap_event_days':
                event_overlap,
            'overlap_intensification_days':
                intensification_overlap,
            'overlap_persistence_days':
                persistence_overlap,

            'event_overlap_fraction_native':
                event_overlap / native_days,

            'intensification_overlap_fraction_native':
                intensification_overlap / native_days,

            'persistence_overlap_fraction_native':
                persistence_overlap / native_days,

            'relative_days_mid_to_onset':
                (midpoint - ev.onset_start_date).days,

            'relative_days_mid_to_event_end':
                (midpoint - ev.event_end_date).days,

            'overlaps_other_primary_event_flag':
                overlaps_other_event(
                    ev.event_id,
                    start,
                    end
                ),

            'ET_primary_mm_composite':
                obs.ET_primary_mm_composite,
            'PET_primary_mm_composite':
                obs.PET_primary_mm_composite,
            'ET_primary_mm_day':
                obs.ET_primary_mm_day,
            'PET_primary_mm_day':
                obs.PET_primary_mm_day,
            'ET_PET_ratio_primary':
                obs.ET_PET_ratio_primary,

            'primary_valid_fraction':
                obs.primary_valid_fraction,
            'primary_data_available_flag':
                obs.primary_data_available_flag,

            'ET_strictQA_mm_composite':
                obs.ET_strictQA_mm_composite,
            'PET_strictQA_mm_composite':
                obs.PET_strictQA_mm_composite,
            'ET_strictQA_mm_day':
                obs.ET_strictQA_mm_day,
            'PET_strictQA_mm_day':
                obs.PET_strictQA_mm_day,
            'ET_PET_ratio_strictQA':
                obs.ET_PET_ratio_strictQA,

            'strict_QA_fraction':
                obs.strict_QA_fraction,
            'strict_QA_data_available_flag':
                obs.strict_QA_data_available_flag,

            'MODLAND_good_fraction':
                obs.MODLAND_good_fraction,
            'clear_pixel_fraction':
                obs.clear_pixel_fraction,
            'SCF_0_or_1_fraction':
                obs.SCF_0_or_1_fraction,

            'context_window_start':
                context_start,
            'context_window_end':
                context_end
        })


mod16_link = pd.DataFrame(mod16_rows)

assert len(mod16_link) > 0
assert set(mod16_link['event_id']) == set(events['event_id'])

print('MOD16 linkage rows:', len(mod16_link))
print(mod16_link['alignment_phase'].value_counts())


## 3. MOD13Q1 native-anchor linkage

Operational context window:
- 48 days before onset
- 96 days after event end

**No MOD13 end date is synthesized.**


In [ ]:
mod13_rows = []

for ev in events.itertuples(index=False):

    context_start = (
        ev.onset_start_date
        - pd.Timedelta(days=48)
    )

    context_end = (
        ev.event_end_date
        + pd.Timedelta(days=96)
    )

    subset = mod13.loc[
        (
            mod13['composite_start_date']
            >= context_start
        )
        &
        (
            mod13['composite_start_date']
            <= context_end
        )
    ].copy()

    for obs in subset.itertuples(index=False):

        anchor = obs.composite_start_date

        if (
            ev.onset_start_date
            <= anchor
            <= ev.threshold_crossing_end_date
        ):
            phase = 'intensification_anchor'

        elif (
            ev.threshold_crossing_end_date
            < anchor
            <= ev.event_end_date
        ):
            phase = 'persistence_anchor'

        elif anchor < ev.onset_start_date:
            phase = 'pre_event_context'

        else:
            phase = 'post_event_context'

        mod13_rows.append({
            'event_id':
                ev.event_id,
            'primary_driver_class':
                ev.primary_driver_class,
            'onset_season':
                ev.onset_season,

            'onset_start_date':
                ev.onset_start_date,
            'threshold_crossing_end_date':
                ev.threshold_crossing_end_date,
            'event_end_date':
                ev.event_end_date,

            'native_composite_start_date':
                anchor,
            'native_system_index':
                obs.native_system_index,
            'nominal_temporal_resolution_days':
                obs.nominal_temporal_resolution_days,

            'alignment_phase':
                phase,

            'relative_days_anchor_to_onset':
                (anchor - ev.onset_start_date).days,

            'relative_days_anchor_to_event_end':
                (anchor - ev.event_end_date).days,

            'anchor_inside_current_event_flag':
                int(
                    ev.onset_start_date
                    <= anchor
                    <= ev.event_end_date
                ),

            'anchor_inside_other_primary_event_flag':
                point_inside_other_event(
                    ev.event_id,
                    anchor
                ),

            'NDVI_allValid':
                obs.NDVI_allValid,
            'EVI_allValid':
                obs.EVI_allValid,
            'all_valid_fraction':
                obs.all_valid_fraction,

            'NDVI_standardQA':
                obs.NDVI_standardQA,
            'EVI_standardQA':
                obs.EVI_standardQA,
            'standard_QA_fraction':
                obs.standard_QA_fraction,
            'standard_QA_data_available_flag':
                obs.standard_QA_data_available_flag,

            'NDVI_strictQA':
                obs.NDVI_strictQA,
            'EVI_strictQA':
                obs.EVI_strictQA,
            'strict_QA_fraction':
                obs.strict_QA_fraction,
            'strict_QA_data_available_flag':
                obs.strict_QA_data_available_flag,

            'SummaryQA_good_fraction':
                obs.SummaryQA_good_fraction,
            'SummaryQA_marginal_fraction':
                obs.SummaryQA_marginal_fraction,

            'context_window_start':
                context_start,
            'context_window_end':
                context_end
        })


mod13_link = pd.DataFrame(mod13_rows)

assert len(mod13_link) > 0
assert set(mod13_link['event_id']) == set(events['event_id'])

print('MOD13 linkage rows:', len(mod13_link))
print(mod13_link['alignment_phase'].value_counts())


## 4. GRACE/GRACE-FO native-interval linkage

Operational context window:
- 180 days before onset
- 365 days after event end

The broad context is retained for later lag analysis. It does **not** imply that a 365-day lag is expected or significant.


In [ ]:
grace_rows = []

for ev in events.itertuples(index=False):

    context_start = (
        ev.onset_start_date
        - pd.Timedelta(days=180)
    )

    context_end = (
        ev.event_end_date
        + pd.Timedelta(days=365)
    )

    subset = grace.loc[
        (
            grace['native_end_date_if_available']
            >= context_start
        )
        &
        (
            grace['native_date']
            <= context_end
        )
    ].copy()

    for obs in subset.itertuples(index=False):

        start = obs.native_date
        end = obs.native_end_date_if_available
        midpoint = start + (end - start) / 2
        native_days = int((end - start).days + 1)

        intensification_overlap = overlap_days(
            start,
            end,
            ev.onset_start_date,
            ev.threshold_crossing_end_date
        )

        persistence_start = (
            ev.threshold_crossing_end_date
            + pd.Timedelta(days=1)
        )

        persistence_overlap = 0

        if persistence_start <= ev.event_end_date:
            persistence_overlap = overlap_days(
                start,
                end,
                persistence_start,
                ev.event_end_date
            )

        event_overlap = (
            intensification_overlap
            + persistence_overlap
        )

        if event_overlap > 0:

            if (
                intensification_overlap > 0
                and persistence_overlap > 0
            ):
                phase = 'mixed_intensification_persistence'

            elif intensification_overlap > 0:
                phase = 'intensification_overlap'

            else:
                phase = 'persistence_overlap'

        elif end < ev.onset_start_date:
            phase = 'pre_event_context'

        else:
            phase = 'post_event_context'

        grace_rows.append({
            'event_id':
                ev.event_id,
            'primary_driver_class':
                ev.primary_driver_class,
            'onset_season':
                ev.onset_season,

            'onset_start_date':
                ev.onset_start_date,
            'threshold_crossing_end_date':
                ev.threshold_crossing_end_date,
            'event_end_date':
                ev.event_end_date,

            'native_system_index':
                obs.native_system_index,
            'native_start_date':
                start,
            'native_end_date':
                end,
            'native_mid_date':
                midpoint,
            'native_interval_days':
                native_days,

            'alignment_phase':
                phase,

            'overlap_event_days':
                event_overlap,
            'overlap_intensification_days':
                intensification_overlap,
            'overlap_persistence_days':
                persistence_overlap,

            'event_overlap_fraction_native':
                event_overlap / native_days,

            'relative_days_mid_to_onset':
                (midpoint - ev.onset_start_date).days,

            'relative_days_mid_to_event_end':
                (midpoint - ev.event_end_date).days,

            'overlaps_other_primary_event_flag':
                overlaps_other_event(
                    ev.event_id,
                    start,
                    end
                ),

            'TWSA_cm':
                obs.TWSA_cm,

            'mean_mascon_uncertainty_cm':
                obs.mean_mascon_uncertainty_cm,

            'TWSA_data_available_flag':
                obs.TWSA_data_available_flag,

            'context_window_start':
                context_start,
            'context_window_end':
                context_end
        })


grace_link = pd.DataFrame(grace_rows)

assert len(grace_link) > 0

print('GRACE linkage rows:', len(grace_link))
print('Events with GRACE context:',
      grace_link['event_id'].nunique())
print(grace_link['alignment_phase'].value_counts())


## 5. JRC native-month linkage

Operational context window:
- 90 days before onset
- 180 days after event end

The month end is derived only for monthly interval-overlap bookkeeping. No sub-monthly JRC values are created.


In [ ]:
jrc_work = jrc.copy()

jrc_work['native_month_start'] = (
    jrc_work['date']
)

jrc_work['native_month_end'] = (
    jrc_work['date']
    + MonthEnd(0)
)

jrc_work['native_month_mid'] = (
    jrc_work['native_month_start']
    +
    (
        jrc_work['native_month_end']
        - jrc_work['native_month_start']
    ) / 2
)

jrc_rows = []

for ev in events.itertuples(index=False):

    context_start = (
        ev.onset_start_date
        - pd.Timedelta(days=90)
    )

    context_end = (
        ev.event_end_date
        + pd.Timedelta(days=180)
    )

    subset = jrc_work.loc[
        (
            jrc_work['native_month_end']
            >= context_start
        )
        &
        (
            jrc_work['native_month_start']
            <= context_end
        )
    ].copy()

    for obs in subset.itertuples(index=False):

        start = obs.native_month_start
        end = obs.native_month_end
        midpoint = obs.native_month_mid
        native_days = int((end - start).days + 1)

        intensification_overlap = overlap_days(
            start,
            end,
            ev.onset_start_date,
            ev.threshold_crossing_end_date
        )

        persistence_start = (
            ev.threshold_crossing_end_date
            + pd.Timedelta(days=1)
        )

        persistence_overlap = 0

        if persistence_start <= ev.event_end_date:
            persistence_overlap = overlap_days(
                start,
                end,
                persistence_start,
                ev.event_end_date
            )

        event_overlap = (
            intensification_overlap
            + persistence_overlap
        )

        if event_overlap > 0:

            if (
                intensification_overlap > 0
                and persistence_overlap > 0
            ):
                phase = 'mixed_intensification_persistence'

            elif intensification_overlap > 0:
                phase = 'intensification_overlap'

            else:
                phase = 'persistence_overlap'

        elif end < ev.onset_start_date:
            phase = 'pre_event_context'

        else:
            phase = 'post_event_context'

        coverage_qualified = int(
            (
                obs.no_valid_observation_flag == 0
            )
            and
            (
                obs.low_coverage_flag == 0
            )
        )

        jrc_rows.append({
            'event_id':
                ev.event_id,
            'primary_driver_class':
                ev.primary_driver_class,
            'onset_season':
                ev.onset_season,

            'onset_start_date':
                ev.onset_start_date,
            'threshold_crossing_end_date':
                ev.threshold_crossing_end_date,
            'event_end_date':
                ev.event_end_date,

            'native_system_index':
                obs.native_system_index,
            'native_month_start':
                start,
            'native_month_end':
                end,
            'native_month_mid':
                midpoint,
            'native_month_days':
                native_days,

            'alignment_phase':
                phase,

            'overlap_event_days':
                event_overlap,
            'overlap_intensification_days':
                intensification_overlap,
            'overlap_persistence_days':
                persistence_overlap,

            'event_overlap_fraction_native':
                event_overlap / native_days,

            'relative_days_mid_to_onset':
                (midpoint - ev.onset_start_date).days,

            'relative_days_mid_to_event_end':
                (midpoint - ev.event_end_date).days,

            'overlaps_other_primary_event_flag':
                overlaps_other_event(
                    ev.event_id,
                    start,
                    end
                ),

            'valid_observed_area_km2':
                obs.valid_observed_area_km2,

            'valid_observation_fraction':
                obs.valid_observation_fraction,

            'water_area_km2':
                obs.water_area_km2,

            'water_fraction_of_valid_area':
                obs.water_fraction_of_valid_area,

            'no_valid_observation_flag':
                obs.no_valid_observation_flag,

            'low_coverage_flag':
                obs.low_coverage_flag,

            'water_detected_flag':
                obs.water_detected_flag,

            'coverage_qualified_flag':
                coverage_qualified,

            'context_window_start':
                context_start,
            'context_window_end':
                context_end
        })


jrc_link = pd.DataFrame(jrc_rows)

assert len(jrc_link) > 0

print('JRC linkage rows:', len(jrc_link))
print('Events with JRC context:',
      jrc_link['event_id'].nunique())
print(jrc_link['alignment_phase'].value_counts())


## 6. Event-level response-data availability matrix


In [ ]:
availability = events[[
    'event_id',
    'onset_pentad',
    'onset_start_date',
    'event_end_date',
    'total_duration_pentads',
    'prolonged_after_flash_flag',
    'primary_driver_class',
    'onset_season'
]].copy()


# -----------------------
# MOD16 availability
# -----------------------

mod16_context = (
    mod16_link
    .groupby('event_id')
    .size()
)

mod16_event = (
    mod16_link.loc[
        mod16_link['overlap_event_days'] > 0
    ]
    .groupby('event_id')
    .size()
)

mod16_strict_event = (
    mod16_link.loc[
        (
            mod16_link['overlap_event_days'] > 0
        )
        &
        (
            mod16_link[
                'strict_QA_data_available_flag'
            ] == 1
        )
    ]
    .groupby('event_id')
    .size()
)

mod16_min_valid = (
    mod16_link.loc[
        mod16_link['overlap_event_days'] > 0
    ]
    .groupby('event_id')[
        'primary_valid_fraction'
    ]
    .min()
)

availability['MOD16_context_records'] = (
    availability['event_id']
    .map(mod16_context)
    .fillna(0)
    .astype(int)
)

availability['MOD16_event_overlap_records'] = (
    availability['event_id']
    .map(mod16_event)
    .fillna(0)
    .astype(int)
)

availability[
    'MOD16_strictQA_event_overlap_records'
] = (
    availability['event_id']
    .map(mod16_strict_event)
    .fillna(0)
    .astype(int)
)

availability[
    'MOD16_min_primary_valid_fraction_event_overlap'
] = (
    availability['event_id']
    .map(mod16_min_valid)
)


# -----------------------
# MOD13 availability
# -----------------------

mod13_event_mask = (
    mod13_link['alignment_phase']
    .isin([
        'intensification_anchor',
        'persistence_anchor'
    ])
)

mod13_context = (
    mod13_link
    .groupby('event_id')
    .size()
)

mod13_event = (
    mod13_link.loc[
        mod13_event_mask
    ]
    .groupby('event_id')
    .size()
)

mod13_standard_event = (
    mod13_link.loc[
        mod13_event_mask
        &
        (
            mod13_link[
                'standard_QA_data_available_flag'
            ] == 1
        )
    ]
    .groupby('event_id')
    .size()
)

mod13_min_standard_fraction = (
    mod13_link.loc[
        mod13_event_mask
    ]
    .groupby('event_id')[
        'standard_QA_fraction'
    ]
    .min()
)

availability['MOD13_context_records'] = (
    availability['event_id']
    .map(mod13_context)
    .fillna(0)
    .astype(int)
)

availability['MOD13_event_anchor_records'] = (
    availability['event_id']
    .map(mod13_event)
    .fillna(0)
    .astype(int)
)

availability[
    'MOD13_standardQA_event_anchor_records'
] = (
    availability['event_id']
    .map(mod13_standard_event)
    .fillna(0)
    .astype(int)
)

availability[
    'MOD13_min_standard_QA_fraction_event_anchors'
] = (
    availability['event_id']
    .map(mod13_min_standard_fraction)
)


# -----------------------
# GRACE availability
# -----------------------

grace_context = (
    grace_link
    .groupby('event_id')
    .size()
)

grace_event = (
    grace_link.loc[
        grace_link['overlap_event_days'] > 0
    ]
    .groupby('event_id')
    .size()
)

availability['GRACE_context_records'] = (
    availability['event_id']
    .map(grace_context)
    .fillna(0)
    .astype(int)
)

availability['GRACE_event_overlap_records'] = (
    availability['event_id']
    .map(grace_event)
    .fillna(0)
    .astype(int)
)

availability['GRACE_context_available_flag'] = (
    availability['GRACE_context_records'] > 0
).astype(int)


# -----------------------
# JRC availability
# -----------------------

jrc_context = (
    jrc_link
    .groupby('event_id')
    .size()
)

jrc_event = (
    jrc_link.loc[
        jrc_link['overlap_event_days'] > 0
    ]
    .groupby('event_id')
    .size()
)

jrc_qualified_event = (
    jrc_link.loc[
        (
            jrc_link['overlap_event_days'] > 0
        )
        &
        (
            jrc_link[
                'coverage_qualified_flag'
            ] == 1
        )
    ]
    .groupby('event_id')
    .size()
)

availability['JRC_context_records'] = (
    availability['event_id']
    .map(jrc_context)
    .fillna(0)
    .astype(int)
)

availability['JRC_event_overlap_records'] = (
    availability['event_id']
    .map(jrc_event)
    .fillna(0)
    .astype(int)
)

availability[
    'JRC_qualified_event_overlap_records'
] = (
    availability['event_id']
    .map(jrc_qualified_event)
    .fillna(0)
    .astype(int)
)

availability['JRC_context_available_flag'] = (
    availability['JRC_context_records'] > 0
).astype(int)


# Combined support flags
availability['all_four_event_overlap_flag'] = (
    (
        availability[
            'MOD16_event_overlap_records'
        ] > 0
    )
    &
    (
        availability[
            'MOD13_event_anchor_records'
        ] > 0
    )
    &
    (
        availability[
            'GRACE_event_overlap_records'
        ] > 0
    )
    &
    (
        availability[
            'JRC_event_overlap_records'
        ] > 0
    )
).astype(int)

availability[
    'all_four_with_JRC_qualified_flag'
] = (
    (
        availability[
            'MOD16_event_overlap_records'
        ] > 0
    )
    &
    (
        availability[
            'MOD13_event_anchor_records'
        ] > 0
    )
    &
    (
        availability[
            'GRACE_event_overlap_records'
        ] > 0
    )
    &
    (
        availability[
            'JRC_qualified_event_overlap_records'
        ] > 0
    )
).astype(int)

assert len(availability) == 39
assert availability['event_id'].is_unique

display(availability.head())


## 7. Phase 2E QC


In [ ]:
qc_rows = []


def add_qc(check, value, status, note):
    qc_rows.append({
        'check': check,
        'value': value,
        'status': status,
        'note': note
    })


add_qc(
    'Primary events',
    len(events),
    'PASS',
    'Must remain 39.'
)

add_qc(
    'MOD16 source rows',
    len(mod16),
    'PASS',
    'Expected 1196.'
)

add_qc(
    'MOD13 source rows',
    len(mod13),
    'PASS',
    'Expected 595.'
)

add_qc(
    'GRACE source rows',
    len(grace),
    'PASS',
    'Expected 238.'
)

add_qc(
    'JRC source rows',
    len(jrc),
    'PASS',
    'Expected 264.'
)

add_qc(
    'MOD16 linkage rows',
    len(mod16_link),
    'RESULT',
    'Native interval links.'
)

add_qc(
    'MOD13 linkage rows',
    len(mod13_link),
    'RESULT',
    'Native anchor links; no synthetic end date.'
)

add_qc(
    'GRACE linkage rows',
    len(grace_link),
    'RESULT',
    'Native irregular interval links.'
)

add_qc(
    'JRC linkage rows',
    len(jrc_link),
    'RESULT',
    'Native monthly links.'
)

add_qc(
    'Events with MOD16 event-overlap records',
    int(
        (
            availability[
                'MOD16_event_overlap_records'
            ] > 0
        ).sum()
    ),
    'PASS',
    'Expected support for all 39 events.'
)

add_qc(
    'Events with MOD13 event anchors',
    int(
        (
            availability[
                'MOD13_event_anchor_records'
            ] > 0
        ).sum()
    ),
    'PASS',
    'Expected support for all 39 events.'
)

add_qc(
    'Events with GRACE context',
    int(
        availability[
            'GRACE_context_available_flag'
        ].sum()
    ),
    'RESULT',
    'Source-era availability; missing is not zero.'
)

add_qc(
    'Events with GRACE direct event overlap',
    int(
        (
            availability[
                'GRACE_event_overlap_records'
            ] > 0
        ).sum()
    ),
    'RESULT',
    'Monthly/irregular sampling limits direct overlap.'
)

add_qc(
    'Events with JRC context',
    int(
        availability[
            'JRC_context_available_flag'
        ].sum()
    ),
    'RESULT',
    'JRC ends in 2021.'
)

add_qc(
    'Events with JRC direct event overlap',
    int(
        (
            availability[
                'JRC_event_overlap_records'
            ] > 0
        ).sum()
    ),
    'RESULT',
    'Monthly source-era support.'
)

add_qc(
    'Events with coverage-qualified JRC event overlap',
    int(
        (
            availability[
                'JRC_qualified_event_overlap_records'
            ] > 0
        ).sum()
    ),
    'RESULT',
    'Low coverage retained but separately flagged.'
)

add_qc(
    'Events with all four datasets directly represented',
    int(
        availability[
            'all_four_event_overlap_flag'
        ].sum()
    ),
    'RESULT',
    'Availability only; not a response-quality score.'
)

add_qc(
    'Events with all four plus qualified JRC overlap',
    int(
        availability[
            'all_four_with_JRC_qualified_flag'
        ].sum()
    ),
    'RESULT',
    'Availability only.'
)

add_qc(
    'JRC low-coverage linkage rows retained',
    int(
        jrc_link[
            'low_coverage_flag'
        ].sum()
    ),
    'PASS',
    'Must be retained, not deleted.'
)

add_qc(
    'JRC no-observation linkage rows retained',
    int(
        jrc_link[
            'no_valid_observation_flag'
        ].sum()
    ),
    'PASS',
    'Must remain missing response, not zero.'
)

add_qc(
    'MOD13 standard-QA unavailable linkage rows retained',
    int(
        (
            mod13_link[
                'standard_QA_data_available_flag'
            ] == 0
        ).sum()
    ),
    'PASS',
    'Retained for QC traceability.'
)

add_qc(
    'MOD16 strict-QA unavailable linkage rows retained',
    int(
        (
            mod16_link[
                'strict_QA_data_available_flag'
            ] == 0
        ).sum()
    ),
    'PASS',
    'Strict QA is a sensitivity track.'
)

qc = pd.DataFrame(qc_rows)

display(qc)

assert len(events) == 39
assert (
    availability[
        'MOD16_event_overlap_records'
    ] > 0
).sum() == 39

assert (
    availability[
        'MOD13_event_anchor_records'
    ] > 0
).sum() == 39


## 8. Save Phase 2E tables automatically


In [ ]:
MOD16_OUT = PHASE2E_DIR / 'AEZ26_Phase2E_MOD16_Native_Response_Links.csv'
MOD13_OUT = PHASE2E_DIR / 'AEZ26_Phase2E_MOD13_Native_Response_Links.csv'
GRACE_OUT = PHASE2E_DIR / 'AEZ26_Phase2E_GRACE_TWSA_Native_Response_Links.csv'
JRC_OUT = PHASE2E_DIR / 'AEZ26_Phase2E_JRC_Monthly_Response_Links.csv'
AVAIL_OUT = PHASE2E_DIR / 'AEZ26_Phase2E_Event_Response_Availability_Summary.csv'
QC_OUT = PHASE2E_DIR / 'AEZ26_Phase2E_QC.csv'
CONFIG_OUT = PHASE2E_DIR / 'AEZ26_Phase2E_Method_Config.txt'

mod16_link.to_csv(
    MOD16_OUT,
    index=False,
    date_format='%Y-%m-%d'
)

mod13_link.to_csv(
    MOD13_OUT,
    index=False,
    date_format='%Y-%m-%d'
)

grace_link.to_csv(
    GRACE_OUT,
    index=False,
    date_format='%Y-%m-%d'
)

jrc_link.to_csv(
    JRC_OUT,
    index=False,
    date_format='%Y-%m-%d'
)

availability.to_csv(
    AVAIL_OUT,
    index=False,
    date_format='%Y-%m-%d'
)

qc.to_csv(
    QC_OUT,
    index=False
)

method_config = '''PHASE 2E METHOD CONFIG
Primary flash-drought catalogue: Phase 2C v3 — 39 events
Flash-drought labels changed in Phase 2E: NO

Event phases:
- intensification = onset_start_date through threshold_crossing_end_date inclusive
- persistence = day after threshold_crossing_end_date through event_end_date inclusive

MOD16:
- native actual composite intervals retained
- context: onset -32 days to event_end +64 days
- exact event-overlap days/fractions calculated
- ET/PET mm/day is normalized composite rate, not a daily observation
- strict QA retained as sensitivity

MOD13:
- native composite_start_date retained
- NO synthesized MOD13 composite end date
- context: onset -48 days to event_end +96 days
- standard QA retained as primary vegetation candidate
- no arbitrary coverage threshold imposed in Phase 2E

GRACE:
- native start/end intervals retained
- context: onset -180 days to event_end +365 days
- no interpolation
- TWSA is terrestrial water-storage anomaly, NOT direct groundwater
- provider uncertainty retained; no formal propagated regional uncertainty calculated

JRC:
- native monthly support retained
- month end derived only for monthly interval bookkeeping
- context: onset -90 days to event_end +180 days
- low-coverage/no-observation records retained
- coverage_qualified_flag requires no_valid_observation_flag=0 and low_coverage_flag=0
- water area is area, NOT storage volume

Context windows are data-extraction windows only; they are NOT inferential response/lag thresholds.

Other-event overlap/confounding flags are retained.
'''

CONFIG_OUT.write_text(
    method_config,
    encoding='utf-8'
)

print('Saved Phase 2E outputs:')
for p in [
    MOD16_OUT,
    MOD13_OUT,
    GRACE_OUT,
    JRC_OUT,
    AVAIL_OUT,
    QC_OUT,
    CONFIG_OUT
]:
    print(' -', p)


## 9. Diagnostic figures

These figures summarize **data support**, not ecological/hydrologic response magnitude.


In [ ]:
# Figure 1 — number of events with direct native response support
support_counts = pd.DataFrame({
    'Dataset': [
        'MOD16',
        'MOD13',
        'GRACE',
        'JRC',
        'JRC qualified'
    ],
    'Events': [
        int(
            (
                availability[
                    'MOD16_event_overlap_records'
                ] > 0
            ).sum()
        ),
        int(
            (
                availability[
                    'MOD13_event_anchor_records'
                ] > 0
            ).sum()
        ),
        int(
            (
                availability[
                    'GRACE_event_overlap_records'
                ] > 0
            ).sum()
        ),
        int(
            (
                availability[
                    'JRC_event_overlap_records'
                ] > 0
            ).sum()
        ),
        int(
            (
                availability[
                    'JRC_qualified_event_overlap_records'
                ] > 0
            ).sum()
        )
    ]
})

plt.figure(figsize=(8, 5))
plt.bar(
    support_counts['Dataset'],
    support_counts['Events']
)
plt.axhline(39)
plt.ylabel('Number of primary flash-drought events')
plt.title('AEZ-26 Native Response-Data Support by Dataset')
plt.xticks(rotation=20)
plt.tight_layout()

SUPPORT_FIG = FIG_DIR / 'AEZ26_Phase2E_Response_Data_Support.png'
plt.savefig(
    SUPPORT_FIG,
    dpi=200,
    bbox_inches='tight'
)
plt.show()


In [ ]:
# Figure 2 — event-by-dataset direct support matrix
matrix = availability[[
    'MOD16_event_overlap_records',
    'MOD13_event_anchor_records',
    'GRACE_event_overlap_records',
    'JRC_event_overlap_records'
]].gt(0).astype(int)

plt.figure(figsize=(8, 10))
plt.imshow(
    matrix.to_numpy(),
    aspect='auto'
)
plt.yticks(
    range(len(availability)),
    availability['onset_pentad']
)
plt.xticks(
    range(4),
    ['MOD16', 'MOD13', 'GRACE', 'JRC']
)
plt.xlabel('Native response dataset')
plt.ylabel('Flash-drought onset pentad')
plt.title('AEZ-26 Event-Level Native Response Availability')
plt.tight_layout()

MATRIX_FIG = FIG_DIR / 'AEZ26_Phase2E_Event_Response_Availability_Matrix.png'
plt.savefig(
    MATRIX_FIG,
    dpi=200,
    bbox_inches='tight'
)
plt.show()


## 10. Create one ZIP audit package

Upload **only this ZIP** to ChatGPT after the notebook completes.


In [ ]:
ZIP_BASE = PHASE2_DIR / 'AEZ26_Phase2E_Output_Package'

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    'zip',
    root_dir=PHASE2_DIR,
    base_dir='Phase2E'
)

print('ZIP CREATED:')
print(zip_path)


In [ ]:
print('FINAL PHASE 2E CHECK')
print('Primary events:', len(events))
print()
print('Linkage rows:')
print('MOD16:', len(mod16_link))
print('MOD13:', len(mod13_link))
print('GRACE:', len(grace_link))
print('JRC:', len(jrc_link))
print()
print('Events with direct event support:')
print(
    'MOD16:',
    int(
        (
            availability[
                'MOD16_event_overlap_records'
            ] > 0
        ).sum()
    )
)
print(
    'MOD13:',
    int(
        (
            availability[
                'MOD13_event_anchor_records'
            ] > 0
        ).sum()
    )
)
print(
    'GRACE:',
    int(
        (
            availability[
                'GRACE_event_overlap_records'
            ] > 0
        ).sum()
    )
)
print(
    'JRC:',
    int(
        (
            availability[
                'JRC_event_overlap_records'
            ] > 0
        ).sum()
    )
)
print(
    'JRC qualified:',
    int(
        (
            availability[
                'JRC_qualified_event_overlap_records'
            ] > 0
        ).sum()
    )
)
print()
print(
    'All four direct:',
    int(
        availability[
            'all_four_event_overlap_flag'
        ].sum()
    )
)
print(
    'All four + qualified JRC:',
    int(
        availability[
            'all_four_with_JRC_qualified_flag'
        ].sum()
    )
)
print()
print('Upload this ZIP for independent audit:')
print(zip_path)


## Next gate

Do **not** calculate final response magnitudes, lag statistics, or groundwater claims before the Phase 2E ZIP is independently audited.

After Phase 2E passes, the next response-analysis stage can derive QC-qualified native-time trajectories, pre-event baselines, lagged response metrics, and sensitivity analyses without destroying the native temporal support.
